Building Synthetic Data Pipeline

In [ ]:
#AI Collision Detection System..
import numpy as np
import pandas as pd
import random

print("GENERATING SATELLITE COLLISION DATASET.....")


SATELLITES = [
    ["ISS", 408, 51.64, "station"],
    ["Starlink-1001", 550, 53.0, "constellation"],
    ["Hubble", 547, 28.47, "telescope"],
    ["Cosmos-1408 Debris", 415, 82.56, "debris"],
    ["Iridium-33 Debris", 780, 86.4, "debris"],
    ["GPS-79", 20200, 55.0, "navigation"],
    ["Envisat", 782, 98.55, "debris"],
    ["Tiangong", 389, 41.47, "station"],
    ["DEBRIS_A", 410, 51.65, "debris"],
    ["DEBRIS_B", 548, 28.46, "debris"]
]

# Generate 5000 random close approach events
np.random.seed(42)
n_samples = 5000

data = []

for i in range(n_samples):
    sat1_idx, sat2_idx = np.random.choice(len(SATELLITES), 2, replace=False)
    sat1 = SATELLITES[sat1_idx]
    sat2 = SATELLITES[sat2_idx]
    
    # Generate realistic features
    distance = np.random.exponential(scale=15)  
    rel_speed = np.random.uniform(0.1, 15)
    alt_diff = abs(sat1[1] - sat2[1])
    inc_diff = abs(sat1[2] - sat2[2])
    
    # Object type risk score
    type_risk = 0.2
    if "Debris" in sat1[0] or "Debris" in sat2[0]:
        type_risk = 0.9
    elif "DEBRIS" in sat1[0] or "DEBRIS" in sat2[0]:
        type_risk = 0.8
    
    # Simulate collision (rare event)
    # Collision if: distance < 0.1km AND relative speed > 5km/s
    collided = 1 if (distance < 0.1 and rel_speed > 5) else 0
    
    # Calculate probability
    prob = 1 / (1 + np.exp(0.5*distance - 0.3*rel_speed + 2*type_risk)) #Sigmoid = 1/1+e^-x
    prob = min(0.99, max(0.01, prob))
    
    data.append({
        'satellite_1': sat1[0],
        'satellite_2': sat2[0],
        'distance_km': round(distance, 3),
        'relative_speed_km_s': round(rel_speed, 3),
        'altitude_diff_km': round(alt_diff, 1),
        'inclination_diff_deg': round(inc_diff, 1),
        'object_type_risk': round(type_risk, 2),
        'time_to_event_hours': np.random.uniform(0.1, 168),
        'collision_occurred': collided,
        'collision_probability': round(prob, 4)
    })

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('satellite_collision_dataset.csv', index=False)

print(f"Dataset created: {len(df)} events")
print(f"Collisions simulated: {df['collision_occurred'].sum()} ({(df['collision_occurred'].sum()/len(df))*100:.1f}%)")
print(" Saved as 'satellite_collision_dataset.csv'")
print("\nFirst 5 rows:")
print(df.head())

🚀 GENERATING SATELLITE COLLISION DATASET
Dataset created: 5000 events
Collisions simulated: 23 (0.5%)
 Saved as 'satellite_collision_dataset.csv'

First 5 rows:
     satellite_1    satellite_2  distance_km  relative_speed_km_s  \
0       DEBRIS_A  Starlink-1001        2.544                0.965   
1            ISS        Envisat        0.012               14.884   
2  Starlink-1001       Tiangong        2.254                4.453   
3  Starlink-1001       DEBRIS_A        8.981                0.298   
4            ISS         GPS-79        1.952                7.478   

   altitude_diff_km  inclination_diff_deg  object_type_risk  \
0               140                   1.4               0.8   
1               374                  46.9               0.2   
2               161                  11.5               0.2   
3               140                   1.4               0.8   
4             19792                   3.4               0.2   

   time_to_event_hours  collision_occurred  c

Building ML Model

In [ ]:

import numpy as np 
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

df = pd.read_csv("satellite_collision_dataset.csv")
fa
